# Fly3D: Video to Gaussian Splat

This notebook runs the low-VRAM Nerfstudio pipeline one stage at a time. Each executable cell streams command output and reports the artifacts it creates.

Use one room per scene. The defaults target an NVIDIA GPU with less than 8 GB VRAM: plain `splatfacto`, 250 frames, three downscale levels, 0.5 training resolution, and 15,000 iterations.

## Prerequisites

Run `./setup.sh` from the project directory first, then select the `fly3d` Conda environment as this notebook's kernel. The environment needs Nerfstudio, COLMAP, FFmpeg, and a working NVIDIA driver.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess

PROJECT_ROOT = Path.cwd()
SCENE_DIR = PROJECT_ROOT / 'scenes' / 'room'
VIDEO_PATH = None  # Set this to a phone video, for example Path.home() / 'Videos/room_4k.mp4'
FRAME_TARGET = 250
NUM_DOWNSCALES = 3
TRAIN_ITERATIONS = 15_000
RESOLUTION_SCALE = 0.5

SCENE_DIR.mkdir(parents=True, exist_ok=True)
print(f'Project: {PROJECT_ROOT}')
print(f'Scene:   {SCENE_DIR}')
print('Set VIDEO_PATH above, then run the cells below in order.')

In [ ]:
def run_streaming(command, cwd=None):
    command = [str(item) for item in command]
    print('$ ' + ' '.join(command))
    process = subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='')
    return_code = process.wait()
    if return_code:
        raise RuntimeError(f'Command failed with exit code {return_code}: {command[0]}')
    return return_code

def show_files(folder, pattern='*'):
    paths = sorted(Path(folder).glob(pattern))
    print(f'\nArtifacts in {folder}: {len(paths)}')
    for path in paths:
        size = f'{path.stat().st_size / 1024 / 1024:.1f} MB' if path.is_file() else 'directory'
        print(f'  {path} ({size})')
    return paths

def require_video():
    if VIDEO_PATH is None:
        raise ValueError('Set VIDEO_PATH in the configuration cell first.')
    video = Path(VIDEO_PATH).expanduser().resolve()
    if not video.is_file():
        raise FileNotFoundError(video)
    return video

## 1. Check the environment and GPU

In [ ]:
commands = ['ns-process-data', 'ns-train', 'ns-export', 'colmap', 'ffmpeg']
for command in commands:
    print(f"{'OK  ' if shutil.which(command) else 'MISS'} {command}")
try:
    gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True, capture_output=True, check=False)
    print('GPU:', gpu.stdout.strip() or gpu.stderr.strip())
except FileNotFoundError:
    print('GPU: nvidia-smi is not installed or the NVIDIA driver is unavailable')

## 2. Ingest the phone video

In [ ]:
video = require_video()
input_video = SCENE_DIR / 'input.mp4'
if input_video.exists() or input_video.is_symlink():
    input_video.unlink()
input_video.symlink_to(video)
print(f'Ingested: {video}')
print(f'Scene input: {input_video} -> {input_video.resolve()}')

## 3. Extract frames and recover camera poses

This is the COLMAP stage. For phone video, sequential matching is faster and uses the natural frame order.

In [ ]:
processed_dir = SCENE_DIR / 'processed'
run_streaming([
    'ns-process-data', 'video',
    '--data', SCENE_DIR / 'input.mp4',
    '--output-dir', processed_dir,
    '--num-frames-target', FRAME_TARGET,
    '--sfm-tool', 'colmap',
    '--matching-method', 'sequential',
    '--num-downscales', NUM_DOWNSCALES,
])
show_files(processed_dir, '**/*')
transforms = processed_dir / 'transforms.json'
print(f'Camera poses ready: {transforms.exists()}')

## 4. Train the Gaussian splat

The Nerfstudio viewer normally appears at `http://localhost:7007` while this cell runs. Stop with the notebook interrupt button if you need to keep the latest checkpoint.

In [ ]:
if not (SCENE_DIR / 'processed' / 'transforms.json').exists():
    raise FileNotFoundError('Run the pose-processing cell first.')
run_streaming([
    'ns-train', 'splatfacto',
    '--data', SCENE_DIR / 'processed',
    '--max-num-iterations', TRAIN_ITERATIONS,
    '--pipeline.datamanager.camera-res-scale-factor', RESOLUTION_SCALE,
])
outputs_dir = SCENE_DIR / 'outputs'
show_files(outputs_dir, '**/*')

## 5. Export `splat.ply`

In [ ]:
configs = sorted((SCENE_DIR / 'outputs').glob('**/config.yml'), key=lambda path: path.stat().st_mtime, reverse=True)
if not configs:
    raise FileNotFoundError('No Nerfstudio config.yml found. Run training first.')
latest_config = configs[0]
exports_dir = SCENE_DIR / 'exports'
run_streaming(['ns-export', 'gaussian-splat', '--load-config', latest_config, '--output-dir', exports_dir])
ply_path = exports_dir / 'splat.ply'
show_files(exports_dir, '*')
print(f'Latest config: {latest_config}')
print(f'PLY ready: {ply_path.exists()}')

## 6. Inspect the final scene artifacts

Open `splat.ply` in SuperSplat for cleanup, orientation, and optional conversion to `.spz` for web delivery.

In [ ]:
summary = {
    'scene': str(SCENE_DIR),
    'input_video': str(SCENE_DIR / 'input.mp4'),
    'processed_poses': (SCENE_DIR / 'processed' / 'transforms.json').exists(),
    'latest_config': str(latest_config) if 'latest_config' in globals() else None,
    'ply': str(SCENE_DIR / 'exports' / 'splat.ply'),
    'ply_exists': (SCENE_DIR / 'exports' / 'splat.ply').exists(),
}
print(json.dumps(summary, indent=2))
print('Next: clean splat.ply in https://superspl.at/editor, then export cleaned.ply or room.spz.')